<a href="https://colab.research.google.com/github/sohnaamie/gambia-semantic-segmentation/blob/amie-deeplab/notebooks/deeplab_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print(device)

In [ ]:
model = torchvision.models.segmentation.deeplabv3_resnet50(
    weights='DEFAULT'
)

model.to(device)

model.eval()

In [ ]:
transform = transforms.Compose([

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
import requests
from PIL import Image
from io import BytesIO

url = "https://images.unsplash.com/photo-1449824913935-59a10b8d2000"

response = requests.get(url)

image = Image.open(
    BytesIO(response.content)
).convert('RGB')

plt.figure(figsize=(10,6))
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
!kaggle datasets list -s IDD

In [ ]:
input_tensor = transform(image)

input_batch = input_tensor.unsqueeze(0).to(device)

with torch.no_grad():

    output = model(input_batch)['out'][0]

prediction = output.argmax(0)

print(np.unique(
    prediction.cpu().numpy()
))

In [ ]:
plt.figure(figsize=(10,8))

plt.imshow(
    prediction.cpu().numpy(),
    cmap='tab20'
)

plt.colorbar()

plt.title(
    "DeepLabV3+ Baseline Prediction"
)

plt.axis('off')

plt.show()

In [ ]:
pred_np = prediction.cpu().numpy()

print(np.unique(pred_np))

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(14,6))

ax[0].imshow(image)
ax[0].set_title("Original Image")
ax[0].axis('off')

ax[1].imshow(prediction.cpu().numpy(), cmap='tab20')
ax[1].set_title("DeepLab Prediction")
ax[1].axis('off')

plt.show()

In [ ]:
scores = torch.softmax(output, dim=0)

print(scores.max())
print(scores.min())

# IDD Dataset Exploration

Goal:
Understand the India Driving Dataset structure, masks, and segmentation labels before training experiments.

In [ ]:
!pip install kaggle

In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = 'amiesohna'
os.environ['KAGGLE_KEY'] = 'export KAGGLE_API_TOKEN=KGAT_80532d130a4b72b78ab4d551e0bd813c'

In [ ]:
!kaggle datasets download -d mitanshuchakrawarty/new-idd-dataset

In [ ]:
!unzip -q new-idd-dataset.zip

replace IDD_RESIZED/image_archive/Image_0.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
for root, dirs, files in os.walk('.'):
    print(root)
    print(dirs[:5])
    print(files[:5])
    print('-'*50)

In [ ]:
sample_id = 100

image_path = f'./IDD_RESIZED/image_archive/Image_{sample_id}.png'

mask_path = f'./IDD_RESIZED/mask_archive/Mask_{sample_id}.png'

image = Image.open(image_path)
mask = Image.open(mask_path)

fig, ax = plt.subplots(
    1,2,
    figsize=(14,6)
)

ax[0].imshow(image)
ax[0].set_title(
    "Correct IDD Image"
)

ax[0].axis('off')

ax[1].imshow(mask)

ax[1].set_title(
    "Correct IDD Mask"
)

ax[1].axis('off')

plt.show()

In [ ]:


mask_np = np.array(mask)

print(mask_np.shape)

print(
np.unique(mask_np)
)

#Creating datset class

In [ ]:

from torch.utils.data import Dataset
import torchvision.transforms as T

class IDDDataset(Dataset):

    def __init__(self, image_dir, mask_dir, transform=None):

        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

        self.images = sorted(os.listdir(image_dir))

    def __len__(self):

        return len(self.images)

    def __getitem__(self, idx):

        image_name = self.images[idx]

        image_path = os.path.join(
            self.image_dir,
            image_name
        )

        mask_name = image_name.replace(
            "Image",
            "Mask"
        )

        mask_path = os.path.join(
            self.mask_dir,
            mask_name
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        mask = Image.open(
            mask_path
        )

        image = T.ToTensor()(image)

        mask = torch.tensor(
            np.array(mask),
            dtype=torch.long
        )

        return image, mask

In [ ]:
#Instantiate Dataset
dataset = IDDDataset(
    "./IDD_RESIZED/image_archive",
    "./IDD_RESIZED/mask_archive"
)

print(len(dataset))

In [ ]:
image, mask = dataset[0]

print(image.shape)

print(mask.shape)

print(torch.unique(mask))

In [ ]:
#creating the dataloader

from torch.utils.data import DataLoader

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

In [ ]:
#Inspect one batch
images, masks = next(iter(dataloader))

print(images.shape)
print(masks.shape)
print(torch.unique(masks))

In [ ]:

import torchvision

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = torchvision.models.segmentation.deeplabv3_resnet50(
    pretrained=True
)

model.to(device)

images = images.to(device)

with torch.no_grad():

    outputs = model(images)['out']

print(outputs.shape)

In [ ]:
predictions = torch.argmax(
    outputs,
    dim=1
)

print(predictions.shape)

print(torch.unique(predictions))

In [ ]:
sample = 0

img = images[sample].cpu().permute(1,2,0)

pred = predictions[sample].cpu()

mask_true = masks[sample].cpu()

fig, ax = plt.subplots(
    1,3,
    figsize=(18,6)
)

ax[0].imshow(img)

ax[0].set_title(
    "Input Image"
)

ax[0].axis('off')

ax[1].imshow(
    pred,
    cmap='tab20'
)

ax[1].set_title(
    "DeepLab Prediction"
)

ax[1].axis('off')

ax[2].imshow(
    mask_true,
    cmap='tab20'
)

ax[2].set_title(
    "Ground Truth Mask"
)

ax[2].axis('off')

plt.show()

Pretrained DeepLab predictions do not align well with ground truth on IDD data.

#Quantitative Evaluation

In [ ]:
#Compute Pixel Accuracy
correct_pixels = (
    predictions == masks.to(device)
).float()

pixel_accuracy = correct_pixels.mean()

print(
"Pixel Accuracy:",
pixel_accuracy.item()
)

In [ ]:
pred_binary = (
    predictions > 0
).long()

print(
torch.unique(pred_binary)
)

In [ ]:
#IoU calculation
intersection = (
    (pred_binary == 1) &
    (masks.to(device) == 1)
).float().sum()

union = (
    (pred_binary == 1) |
    (masks.to(device) == 1)
).float().sum()

iou = intersection / (
    union + 1e-8
)

print(
"IoU:",
iou.item()
)

In [ ]:
#split datset

from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size]
)

print(len(train_dataset))
print(len(val_dataset))

In [ ]:
#create loaders

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False
)

In [ ]:
#modify Deeplab for the output layer

import torchvision
import torch.nn as nn

model = torchvision.models.segmentation.deeplabv3_resnet50(
    pretrained=True
)

model.classifier[4] = nn.Conv2d(
    256,
    2,
    kernel_size=1
)

model.to(device)

In [ ]:
#Define loss function and optimizer

import torch.optim as optim
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [ ]:
# mini training batch

model.train()

images, masks = next(iter(train_loader))

images = images.to(device)
masks = masks.to(device)

optimizer.zero_grad()

outputs = model(images)['out']

print(
"Output Shape:",
outputs.shape
)

loss = criterion(
    outputs,
    masks.long()
)

loss.backward()

optimizer.step()

print(
"Training Loss:",
loss.item()
)

# Full DeepLabV3+ Training Loop

In [ ]:
# Short Training Test

num_epochs = 1
max_batches = 20

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    for batch_idx, (images, masks) in enumerate(train_loader):

        if batch_idx >= max_batches:
            break

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)['out']

        loss = criterion(
            outputs,
            masks.long()
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / max_batches

    print(
        f"Epoch {epoch+1}"
    )

    print(
        "Average Loss:",
        avg_loss
    )

In [ ]:
#Validation Evaluation
model.eval()

total_correct = 0
total_pixels = 0

with torch.no_grad():

    for images, masks in val_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)['out']

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        total_correct += (
            predictions == masks
        ).sum().item()

        total_pixels += masks.numel()

pixel_accuracy = (
    total_correct /
    total_pixels
)

print(
"Validation Pixel Accuracy:",
pixel_accuracy
)

In [ ]:
#Validation IoU Evaluation

model.eval()

intersection = 0
union = 0

with torch.no_grad():

    for images, masks in val_loader:

        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)['out']

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        pred_fg = (predictions == 1)
        true_fg = (masks == 1)

        intersection += (
            pred_fg & true_fg
        ).sum().item()

        union += (
            pred_fg | true_fg
        ).sum().item()

iou = intersection / (
    union + 1e-8
)

print(
"Validation IoU:",
iou
)

# PURPOSE:
# Evaluate segmentation overlap quality using IoU metric.
# Compare predicted foreground regions with ground-truth masks.

In [ ]:
#Controlled Full Training

num_epochs = 3
max_batches = 100

train_losses = []

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    for batch_idx, (images, masks) in enumerate(train_loader):

        if batch_idx >= max_batches:
            break

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)['out']

        loss = criterion(
            outputs,
            masks.long()
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / max_batches

    train_losses.append(
        avg_loss
    )

    print(
        f"Epoch {epoch+1}/{num_epochs}"
    )

    print(
        "Average Loss:",
        avg_loss
    )